# Variant E 100M on a Single GH200

This notebook is separate from the Kaggle launcher. It is tuned for one NVIDIA GH200 on Lambda:

- real `flash-linear-attention` GDN instead of the fallback block
- PyTorch flash SDPA for the dense GQA path
- bf16 autocast on Hopper
- single-GPU execution, no DDP and no DataParallel
- auto-tuned DataLoader workers
- checkpoint resume from the local output dir, with optional Hugging Face mirror support

Run Cell 2 first, then Cell 4 for a fresh launch or Cell 5 to resume from a checkpoint.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:256")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import torch

PROJECT_ROOT: Path | None = None


def _ensure_packages() -> None:
    required = ["pretty_midi", "safetensors", "miditok", "ncps", "huggingface_hub"]
    missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing,
        ])


def _find_project_root() -> Path:
    anchor = Path("training/train_variant_e_100m_ddp.py")
    probes = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    probes.extend([
        Path.home(),
        Path("/workspace"),
        Path("/workspace/piano_midi_model"),
        Path("/mnt"),
        Path("/mnt/data"),
        Path("/data"),
        Path("/datasets"),
    ])
    seen = set()
    for probe in probes:
        key = str(probe)
        if key in seen:
            continue
        seen.add(key)
        if not probe.exists():
            continue
        for candidate in [probe, *probe.parents]:
            if (candidate / anchor).exists():
                return candidate.resolve()
    raise FileNotFoundError(
        "Project root not found; run the notebook from the piano_midi_model workspace or set the working directory so training/train_variant_e_100m_ddp.py is visible."
    )


def _configure_hopper_runtime() -> None:
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        try:
            torch.backends.cudnn.allow_tf32 = True
        except Exception:
            pass
        torch.backends.cudnn.benchmark = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
        try:
            torch.set_autocast_dtype("cuda", torch.bfloat16)
        except Exception:
            try:
                torch.set_autocast_gpu_dtype(torch.bfloat16)
            except Exception:
                pass
        try:
            torch.backends.cuda.enable_flash_sdp(True)
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            torch.backends.cuda.enable_math_sdp(False)
        except Exception:
            pass


def _ensure_real_gdn(require_real_gdn: bool = True) -> None:
    from model.blocks import gdn_block

    if gdn_block.GDN_AVAILABLE:
        print("Real GDN kernels already available.")
        return

    install_specs = ["flash-linear-attention", "flash-linear-attention==0.4.2"]
    last_error = None
    for spec in install_specs:
        try:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                spec,
            ])
            importlib.invalidate_caches()
            if gdn_block.try_import_fla():
                print(f"Installed {spec} and loaded real GDN kernels.")
                return
            print(f"Install succeeded for {spec} but GDN import is still unavailable.")
        except Exception as exc:
            last_error = exc
            print(f"{spec} install failed: {exc}")

    if require_real_gdn:
        raise RuntimeError(
            "flash-linear-attention/GDN backend unavailable in strict mode after install attempts. Restart the kernel and rerun Cell 2, or disable strict mode if you explicitly want fallback GDN."
        ) from last_error


def _read_json_file(path: Path):
    return json.loads(path.read_text(encoding="utf-8-sig"))


def _discover_manifest_candidates() -> list[Path]:
    bases = []
    if PROJECT_ROOT is not None:
        bases.extend([PROJECT_ROOT, PROJECT_ROOT.parent])
    bases.extend([
        Path.cwd().resolve(),
        Path.home(),
        Path("/workspace"),
        Path("/workspace/data"),
        Path("/mnt"),
        Path("/mnt/data"),
        Path("/data"),
        Path("/datasets"),
    ])
    relative_candidates = [
        Path("pulse88_tokenize_500k_work") / "_controller" / "manifest_500k.json",
        Path("pulse88_tokenize_500k_work") / "work" / "manifest.json",
        Path("processed_pretokenized") / "manifest.json",
        Path("processed") / "manifest.json",
        Path("data") / "manifest.json",
        Path("data") / "processed_pretokenized" / "manifest.json",
        Path("manifest_500k.json"),
        Path("manifest.json"),
    ]
    results = []
    seen = set()
    for base in bases:
        if not base.exists():
            continue
        for rel in relative_candidates:
            candidate = base / rel
            key = str(candidate.resolve()) if candidate.exists() else str(candidate)
            if key in seen:
                continue
            if candidate.exists() and candidate.is_file():
                seen.add(key)
                results.append(candidate.resolve())
    return results


def _infer_root_from_manifest(manifest_path: Path) -> Path | None:
    parent = manifest_path.parent
    if parent.name == "_controller" and parent.parent.exists():
        return parent.parent.resolve()
    return parent.resolve() if parent.exists() else None


def _discover_pretokenized_manifest() -> tuple[Path, Path | None]:
    env_manifest = str(os.environ.get("PRETOKENIZED_MANIFEST", "")).strip()
    env_root = str(os.environ.get("PRETOKENIZED_ROOT", "")).strip()
    if env_manifest:
        manifest = Path(env_manifest).expanduser().resolve()
        root = Path(env_root).expanduser().resolve() if env_root else _infer_root_from_manifest(manifest)
        return manifest, root

    candidates = _discover_manifest_candidates()
    if not candidates:
        raise FileNotFoundError(
            "Could not find a pretokenized manifest. Set PRETOKENIZED_MANIFEST and, if needed, PRETOKENIZED_ROOT in the notebook environment before running Cell 4."
        )

    manifest = candidates[0]
    root = _infer_root_from_manifest(manifest)
    return manifest, root


def _gpu_summary() -> str:
    if not torch.cuda.is_available():
        return "CUDA unavailable"
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    return f"{name} (sm_{major}{minor})"


_ensure_packages()
PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
_configure_hopper_runtime()
_ensure_real_gdn(require_real_gdn=True)

from hf_sync import normalize_hf_repo_id, resolve_latest_hf_checkpoint

print(f"Project root: {PROJECT_ROOT}")
print(f"GPU: {_gpu_summary()}")
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    if major < 9:
        print("Warning: this notebook is tuned for Hopper / GH200. You are not on a Hopper-class GPU.")
    try:
        flash_enabled = torch.backends.cuda.flash_sdp_enabled()
        mem_enabled = torch.backends.cuda.mem_efficient_sdp_enabled()
        math_enabled = torch.backends.cuda.math_sdp_enabled()
        print(f"SDPA backends: flash={flash_enabled} mem_efficient={mem_enabled} math={math_enabled}")
    except Exception:
        pass

In [ ]:
import math

MAX_PIECES = 500_000
SEED = 42
EPOCHS = 16
BATCH_PER_GPU = 8
TARGET_EFFECTIVE_BATCH = 64
GRAD_ACCUMULATION_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_BATCH / max(1, BATCH_PER_GPU)))
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.015
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.02
LABEL_SMOOTHING_START = 0.0
LABEL_SMOOTHING_WARMUP_STEPS = 2000
SEED_LENGTH = 512
CONTINUATION_LENGTH = 1536
MAX_SEQUENCE_LENGTH = SEED_LENGTH + CONTINUATION_LENGTH
VOCAB_SIZE = 374
EVENT_SIZE = 4
TOKENIZATION_STRATEGY = "custom_delta"
SAVE_EVERY_N_STEPS = 500
NUM_WORKERS = -1
RUN_TRAINING = True
RUN_RESUME_TRAINING = True
ALLOW_AUTO_RESUME = True

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "variant_e_100m_lambda_gh200"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_FALLBACK = OUTPUT_DIR / "source_pretokenized" / "manifest.json"

PRETOKENIZED_MANIFEST, PRETOKENIZED_ROOT = _discover_pretokenized_manifest()
if not PRETOKENIZED_MANIFEST.exists():
    raise FileNotFoundError(PRETOKENIZED_MANIFEST)

HF_SYNC_REPO_ID = normalize_hf_repo_id(str(os.environ.get("HF_SYNC_REPO_ID", "")).strip())
HF_SYNC_PRIVATE = str(os.environ.get("HF_SYNC_PRIVATE", "1")).strip().lower() in {"1", "true", "yes", "on"}
HF_SYNC_EVERY_N_STEPS = int(os.environ.get("HF_SYNC_EVERY_N_STEPS", str(SAVE_EVERY_N_STEPS)))
HF_TOKEN = str(os.environ.get("HF_TOKEN", "")).strip()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

ddp_script = PROJECT_ROOT / "training" / "train_variant_e_100m_ddp.py"
if not ddp_script.exists():
    raise FileNotFoundError(ddp_script)


def _launch_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    return env


def _build_training_command(*, resume_from_checkpoint: str = "", resume_mode: str = "remaining", auto_resume: bool = True) -> list[str]:
    cmd = [
        sys.executable,
        str(ddp_script),
        "--pretokenized_manifest", str(PRETOKENIZED_MANIFEST),
        "--output_dir", str(OUTPUT_DIR),
        "--max_pieces", str(int(MAX_PIECES)),
        "--seed", str(int(SEED)),
        "--seed_length", str(int(SEED_LENGTH)),
        "--continuation_length", str(int(CONTINUATION_LENGTH)),
        "--max_sequence_length", str(int(MAX_SEQUENCE_LENGTH)),
        "--tokenization_strategy", TOKENIZATION_STRATEGY,
        "--event_size", str(int(EVENT_SIZE)),
        "--vocab_size", str(int(VOCAB_SIZE)),
        "--d_model", "1024",
        "--n_layers", "14",
        "--attention_every_n_layers", "2",
        "--gdn_inner_ratio", "0.5",
        "--gdn_num_heads", "4",
        "--gqa_num_heads", "8",
        "--gqa_groups", "4",
        "--full_attention",
        "--epochs", str(int(EPOCHS)),
        "--batch_size", str(int(BATCH_PER_GPU)),
        "--grad_accumulation_steps", str(int(GRAD_ACCUMULATION_STEPS)),
        "--learning_rate", str(float(LEARNING_RATE)),
        "--warmup_ratio", str(float(WARMUP_RATIO)),
        "--weight_decay", str(float(WEIGHT_DECAY)),
        "--label_smoothing", str(float(LABEL_SMOOTHING)),
        "--label_smoothing_start", str(float(LABEL_SMOOTHING_START)),
        "--label_smoothing_warmup_steps", str(int(LABEL_SMOOTHING_WARMUP_STEPS)),
        "--num_workers", str(int(NUM_WORKERS)),
        "--log_every_n_steps", "20",
        "--save_every_n_steps", str(int(SAVE_EVERY_N_STEPS)),
        "--save_every_n_epochs", "1",
        "--use_amp",
    ]
    if PRETOKENIZED_ROOT is not None:
        cmd.extend(["--pretokenized_root", str(PRETOKENIZED_ROOT)])
    if HF_SYNC_REPO_ID:
        cmd.extend([
            "--hf_sync_repo_id", HF_SYNC_REPO_ID,
            "--hf_sync_every_n_steps", str(int(HF_SYNC_EVERY_N_STEPS)),
        ])
        if HF_SYNC_PRIVATE:
            cmd.append("--hf_sync_private")
    if not auto_resume:
        cmd.append("--no_auto_resume")
    if resume_from_checkpoint:
        cmd.extend(["--resume_from_checkpoint", str(Path(resume_from_checkpoint)), "--resume_mode", resume_mode])
    return cmd


def _find_local_resume_checkpoint() -> Path | None:
    checkpoint_dir = OUTPUT_DIR / "checkpoints" / "variant_e_100m_ddp"
    for candidate in [
        checkpoint_dir / "latest_state.pt",
        checkpoint_dir / "best_state.pt",
        checkpoint_dir / "latest.safetensors",
        checkpoint_dir / "best.safetensors",
    ]:
        if candidate.exists():
            return candidate.resolve()
    return None


print(f"Pre-tokenized manifest: {PRETOKENIZED_MANIFEST}")
print(f"Pre-tokenized root: {PRETOKENIZED_ROOT or '(loader will infer)'}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Fresh launch batch={BATCH_PER_GPU}, grad_accum={GRAD_ACCUMULATION_STEPS}, effective_batch={BATCH_PER_GPU * GRAD_ACCUMULATION_STEPS}")
if HF_SYNC_REPO_ID:
    print(f"HF sync repo: {HF_SYNC_REPO_ID} (private={HF_SYNC_PRIVATE}, every_n_steps={HF_SYNC_EVERY_N_STEPS})")

In [ ]:
if RUN_TRAINING:
    cmd = _build_training_command(auto_resume=ALLOW_AUTO_RESUME)
    print("Launching Variant E 100M training on GH200:")
    print(" ".join(str(part) for part in cmd))
    subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT), env=_launch_env())
else:
    print("Set RUN_TRAINING = True to launch a fresh or auto-resumed run.")

result_json = OUTPUT_DIR / "variant_e_100m_ddp_result.json"
if result_json.exists():
    payload = _read_json_file(result_json)
    print(f"Result JSON: {result_json}")
    print(json.dumps(payload, indent=2))
else:
    print(f"No result JSON yet: {result_json}")

In [ ]:
if RUN_RESUME_TRAINING:
    resume_ckpt = str(_find_local_resume_checkpoint() or "").strip()
    if not resume_ckpt and HF_SYNC_REPO_ID:
        remote_resume = resolve_latest_hf_checkpoint(
            repo_id=HF_SYNC_REPO_ID,
            cache_root=OUTPUT_DIR,
            token=HF_TOKEN,
            repo_type="model",
        )
        if remote_resume is not None:
            resume_ckpt = str(remote_resume.resolve())

    if resume_ckpt:
        cmd = _build_training_command(
            resume_from_checkpoint=resume_ckpt,
            resume_mode="remaining",
            auto_resume=False,
        )
        print("Resuming Variant E 100M from checkpoint:")
        print(resume_ckpt)
        print(" ".join(str(part) for part in cmd))
        subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT), env=_launch_env())
    else:
        print("No local or mirrored checkpoint found yet. Run Cell 4 first.")
else:
    print("Set RUN_RESUME_TRAINING = True to resume from the latest checkpoint.")

result_json = OUTPUT_DIR / "variant_e_100m_ddp_result.json"
if result_json.exists():
    payload = _read_json_file(result_json)
    result = payload.get("result", {}) if isinstance(payload, dict) else {}
    losses = result.get("val_loss", []) if isinstance(result, dict) else []
    if losses:
        print(f"Best val loss: {min(float(v) for v in losses):.6f}")
else:
    print(f"Result JSON not found yet: {result_json}")